# MGS-5 : Construire les métaheuristiques composées — des primitives au raccourci factory

[<- MGS-4 Islands](MGS-4-Islands.ipynb) | [^ Série MGS](README.md) | [MGS-6 Benchmarks ->](MGS-6-Benchmarks.ipynb)

Jusqu'ici (MGS-1 à MGS-4) nous avons vu le moteur `MetaGeneticAlgorithm`, la grammaire fluente de primitives (`Match`, `IfElse`, `Container`, `Scoped`...), et deux composés structurants (Eukaryote, Islands). Ce notebook répond à la question qui reste : **d'où viennent les algorithmes « publiés »** — Whale Optimization Algorithm (WOA), Equilibrium Optimizer (EO), Forensic-Based Investigation (FBI) — et **comment les utiliser sans retomber dans le piège de la boîte noire**.

L'argument de MetaGeneticSharp (cf. MGS-6 Benchmarks) est que reconstruire ces algorithmes à partir de primitives composables *démontre* leur mécanisme au lieu de le cacher derrière une métaphore animale (critique de Sorensen, 2015 : les métaheuristiques bio-inspirées seraient des « metaphors exposed »). Ce notebook montre, pour **chaque métaheuristique composée du catalogue**, le même parcours en trois temps :

1. **La description** — la métaphore et la mécanique, dans l'esprit des fiches de `mealpy` (la bibliothèque Python qui sert de repère, et dont le fork s'inspire explicitement).
2. **La reconstruction depuis les primitives** — le code réel de la lib qui assemble l'algorithme à partir de briques (`IfElse`, `Match`, `GenerationMetaHeuristic`, croisements géométriques...). Ce n'est pas une paraphrase : c'est le corps de la méthode `BuildMainHeuristic()` de la classe, lu transparentement.
3. **Le raccourci factory** — le one-liner `MetaHeuristicsService.CreateMetaHeuristicByName("...")` qui produit *exactement* cette reconstruction, et la preuve (le type racine renvoyé) que raccourci et reconstruction sont identiques.

> **Rappel C.2** : kernel `.net-csharp`. Les DLLs MetaGeneticSharp + GeneticSharp sont chargées depuis le build du fork, compilé en **net9.0** pour correspondre au kernel (`.NET 8` via dotnet-interactive). Voir la cellule de câblage ci-dessous.


In [1]:
// Câblage : chargement des DLLs MetaGeneticSharp + GeneticSharp (build net9.0 du fork).
// Prérequis de build (depuis la racine du fork) :
//   dotnet build src/MetaGeneticSharp.Domain/MetaGeneticSharp.Domain.csproj -c Debug
// (net9.0 pour matcher le target net9.0 du csproj ; les DLLs core + dépendances System.Drawing
//  sont co-localisées dans Domain/bin/Debug/net9.0/.)
#r "c:/dev/MetaGeneticSharp/src/MetaGeneticSharp.Domain/bin/Debug/net9.0/GeneticSharp.Infrastructure.Framework.dll"
#r "c:/dev/MetaGeneticSharp/src/MetaGeneticSharp.Domain/bin/Debug/net9.0/GeneticSharp.Domain.dll"
#r "c:/dev/MetaGeneticSharp/src/MetaGeneticSharp.Domain/bin/Debug/net9.0/MetaGeneticSharp.Infrastructure.dll"
#r "c:/dev/MetaGeneticSharp/src/MetaGeneticSharp.Domain/bin/Debug/net9.0/MetaGeneticSharp.Domain.dll"

using MetaGeneticSharp;
using GeneticSharp;
using System;
using System.Collections.Generic;
using System.Linq;

Console.WriteLine("MetaGeneticSharp + GeneticSharp chargés (build net9.0 du fork).");
Console.WriteLine("  MetaHeuristicsService : " + typeof(MetaHeuristicsService).Name);
Console.WriteLine("  WhaleOptimisationAlgorithm : " + typeof(WhaleOptimisationAlgorithm).Name);
Console.WriteLine("  EquilibriumOptimizer : " + typeof(EquilibriumOptimizer).Name);
Console.WriteLine("  ForensicBasedInvestigation : " + typeof(ForensicBasedInvestigation).Name);


MetaGeneticSharp + GeneticSharp chargés (build net9.0 du fork).


  MetaHeuristicsService : MetaHeuristicsService


  WhaleOptimisationAlgorithm : WhaleOptimisationAlgorithm


  EquilibriumOptimizer : EquilibriumOptimizer


  ForensicBasedInvestigation : ForensicBasedInvestigation


## Le catalogue : `KnownCompoundMetaheuristics`

Le point d'entrée de la factory est une simple énumération : `KnownCompoundMetaheuristics`. Chaque valeur nomme une métaheuristique composée que `MetaHeuristicsService` sait construire. C'est l'équivalent du registre de `mealpy` — une liste close d'algorithmes prêts à l'emploi.

Le catalogue couvre quatre familles :

| Famille | Membres | Idée |
|---------|---------|------|
| **Default** | `Default`, `DefaultRandomHyperspeed` | Le GA GeneticSharp nu (référence). |
| **Géométriques** | `WhaleOptimisation`, `WhaleOptimisationNaive`, `EquilibriumOptimizer`, `ForensicBasedInvestigation` | Composés reconstruits depuis des croisements géométriques sur gènes continus. |
| **Îles homogènes** | `Islands5Default`, `Islands5DefaultNoMigration` | Archipel de 5 îles identiques (cf. MGS-4). |
| **Îles hétérogènes** | `Islands5BestMixture`, `Islands5BestMixtureNoMigration` | Archipel mêlant WOA + EO + GA (composition de composés). |

Listons-le dynamiquement pour confirmer ce que la lib expose *réellement* (plutôt que de faire confiance à une liste écrite à la main).


In [2]:
// Le catalogue vu par la lib (Reflection sur l'enum KnownCompoundMetaheuristics).
var names = MetaHeuristicsService.GetMetaHeuristicNames();
Console.WriteLine("Catalogue KnownCompoundMetaheuristics (" + names.Count + " entrées) :");
foreach (var n in names) Console.WriteLine("  - " + n);

// La factory : un seul appel construit n'importe quel composé par son nom.
// Signature : CreateMetaHeuristicByName(name, maxGenerations=1000, populationSize=100,
//                                       geometricConverter=null, noMutation=true)
IMetaHeuristic demo = MetaHeuristicsService.CreateMetaHeuristicByName("Default");
Console.WriteLine();
Console.WriteLine("CreateMetaHeuristicByName(Default) -> " + demo.GetType().Name);


Catalogue KnownCompoundMetaheuristics (11 entrées) :


  - None


  - Default


  - DefaultRandomHyperspeed


  - WhaleOptimisation


...

  - Islands5Default


  - Islands5DefaultNoMigration


  - Islands5BestMixture


  - Islands5BestMixtureNoMigration


CreateMetaHeuristicByName(Default) -> DefaultMetaHeuristic


## Le terrain commun : chromosome continu et banc d'essai

Les composés géométriques (WOA, EO, FBI) opèrent sur des gènes **continus** lus comme des `double` via un `GeometricConverter`. Le chromosome `DoubleArrayChromosome` ci-dessous (réplique minimale de l'assistant de test du fork) stocke des `double` nus : la conversion gene <-> double est donc l'identité. C'est la représentation commune sur laquelle nous lancerons chaque composé.

Les fonctions test (`Sphere`, `Rastrigin`, `Rosenbrock`) sont définies **inline** plus bas pour garder ce notebook autonome (trois surfaces canoniques d'optimisation continue). Rappel : GeneticSharp *maximise*, les fonctions test *minimisent* -> la fitness est l'objectif **négativé** (plus proche de 0 = meilleur). Le banc d'essai `RunBenchmark` câble un composé dans le moteur `MetaGeneticAlgorithm` (sélection élitiste, croisement uniforme, mutation uniforme) et renvoie la meilleure fitness. `BuildViaFactory(name)` est le raccourci paramétré par nom que nous réutiliserons pour chaque composé.


In [3]:
// DoubleArrayChromosome : chromosome minimal stockant des gènes double nus.
// (Inspiré de l'assistant de test MetaGeneticSharp.Domain.Tests.Geometric.DoubleArrayChromosome,
//  étendu avec des bornes par gène pour que CreateNew() randomise la population initiale.)
public class DoubleArrayChromosome : ChromosomeBase
{
    private readonly double _min;
    private readonly double _max;
    public DoubleArrayChromosome(double[] values, double min, double max) : base(values.Length)
    {
        _min = min; _max = max;
        for (int i = 0; i < values.Length; i++) ReplaceGene(i, new Gene(values[i]));
    }
    public override IChromosome CreateNew()
    {
        var rand = RandomizationProvider.Current;
        int n = Length;
        var vals = new double[n];
        for (int i = 0; i < n; i++) vals[i] = rand.GetDouble(_min, _max);
        return new DoubleArrayChromosome(vals, _min, _max);
    }
    public override Gene GenerateGene(int geneIndex)
        => new Gene(RandomizationProvider.Current.GetDouble(_min, _max));
    public double[] GetDoubleValues() => GetGenes().Select(g => (double)g.Value).ToArray();
}

// --- Fonctions test inline (objectif à minimiser -> fitness = -objectif). ---
// Sphere :    f(x) = Σ x_i²               minimum global 0 en 0.
// Rastrigin : f(x) = Σ (x_i² - 10 cos(2π x_i)) + 10·n   minimum global 0 en 0.
// Rosenbrock: f(x) = Σ [100(x_{i+1}-x_i²)² + (1-x_i)²]  minimum global 0 en (1,...,1).

public class SphereFitness : IFitness
{
    public double Evaluate(IChromosome c)
    {
        var x = ((DoubleArrayChromosome)c).GetDoubleValues();
        double s = 0; for (int i = 0; i < x.Length; i++) s += x[i] * x[i];
        return -s;
    }
}

public class RastriginFitness : IFitness
{
    public double Evaluate(IChromosome c)
    {
        var x = ((DoubleArrayChromosome)c).GetDoubleValues();
        double s = 0;
        for (int i = 0; i < x.Length; i++) s += x[i] * x[i] - 10 * Math.Cos(2 * Math.PI * x[i]);
        return -(s + 10 * x.Length);
    }
}

public class RosenbrockFitness : IFitness
{
    public double Evaluate(IChromosome c)
    {
        var x = ((DoubleArrayChromosome)c).GetDoubleValues();
        double s = 0;
        for (int i = 0; i < x.Length - 1; i++)
        {
            double a = 100 * (x[i + 1] - x[i] * x[i]);
            double b = 1 - x[i];
            s += a * a + b * b;
        }
        return -s;
    }
}

// Bornes usuelles par fonction test (intervalle de tirage de la population initiale).
static (double min, double max) Bounds(Type t) =>
    t == typeof(RosenbrockFitness) ? (-2.048, 2.048) : (-5.12, 5.12);

const int Dim = 5;
const int PopSize = 60;
const int Generations = 80;

// Banc d'essai : lance une métaheuristique sur une fonction test, renvoie la meilleure fitness.
double RunBenchmark(IFitness fitness, Type fitnessType, IMetaHeuristic mh, int dim = Dim)
{
    var (min, max) = Bounds(fitnessType);
    double mid = 0.5 * (min + max);
    var adam = new DoubleArrayChromosome(Enumerable.Repeat(mid, dim).ToArray(), min, max);
    var pop = new MetaPopulation(PopSize, PopSize, adam);
    var ga = new MetaGeneticAlgorithm(
        pop, fitness,
        new EliteSelection(),
        new UniformCrossover(0.5f),
        new UniformMutation(true),
        mh);
    ga.Termination = new GenerationNumberTermination(Generations);
    ga.Start();
    return ga.BestChromosome.Fitness.HasValue ? ga.BestChromosome.Fitness.Value : double.NaN;
}

// Raccourci factory paramétré par nom (câble automatiquement un convertisseur identité pour les géométriques).
IMetaHeuristic BuildViaFactory(string name) =>
    MetaHeuristicsService.CreateMetaHeuristicByName(name, maxGenerations: Generations, populationSize: PopSize);

var demoFns = new (string Name, Type Type)[]
{
    ("Sphere",     typeof(SphereFitness)),
    ("Rastrigin",  typeof(RastriginFitness)),
    ("Rosenbrock", typeof(RosenbrockFitness)),
};
Console.WriteLine("Harness + fonctions test inline + factory BuildViaFactory(name) prêts.");


Harness + fonctions test inline + factory BuildViaFactory(name) prêts.


## Le parcours en trois temps

Pour chaque composé géométrique (WOA, EO, FBI), la lib définit une classe qui hérite de `GeometricMetaHeuristicBase`. Cette classe implémente `BuildMainHeuristic()` : c'est la **reconstruction depuis les primitives** — la méthode qui assemble l'algorithme avec la grammaire fluente vue en MGS-2. Appeler `.Build()` (ou la factory) exécute cette reconstruction et renvoie l'arbre de primitives prêt à l'emploi.

Nous allons donc, pour chaque composé :

- lire sa fiche (métaphore + équations),
- **lire sa reconstruction** `BuildMainHeuristic()` — c'est l'implémentation, montrée transparentement,
- appeler la factory et **prouver** qu'elle renvoie le même type racine que la reconstruction (donc : même arbre).

Commençons par le plus célèbre : WOA.


## 1. Whale Optimization Algorithm (WOA)

### Description

WOA (Mirjalili & Lewis, 2016) mime la **chasse en *bubble-net*** des baleines à bosse. La mécanique se décompose en trois comportements sélectionnés stochastiquement à chaque génération :

- **Encerclement de la proie** : chaque baleine se rapproche de la meilleure solution connue (ou d'une solution aléatoire en phase d'exploration) par un déplacement linéaire `D = |C·x_best - x|` puis `x <- x_best - A·D`.
- **Spirale *bubble-net*** (exploitation) : la baleine décrit une hélice autour de la meilleure proie : `x <- D'·exp(b·l)·cos(2π·l) + x_best`, avec `b` l'amplitude hélicoïdale et `l` un angle aléatoire.
- **Recherche aléatoire** : quand `|A| > 1`, on se déplace vers un individu aléatoire (exploration globale).

Deux paramètres clef : **`a`** décroit linéairement de 2 à 0 (il pilote la bascule exploration -> exploitation via `A = 2a·r - a`), et **`b`** fixe la spirale. C'est exactement la fiche `mealpy` (`mealpy.evolutionary_based.WOA`), dont le fork reproduit les équations.

### Reconstruction depuis les primitives

La classe `WhaleOptimisationAlgorithm.BuildMainHeuristic()` construit l'algorithme ainsi (code réel de la lib, annoté) :

```csharp
var woa = new IfElseMetaHeuristic()                 // tirage < 0.5 -> branche "poursuite", sinon -> bubble-net
    .WithScope(EvolutionStage.Crossover)            // on intercepte le STAGE croisement
    .WithParam("a",  ..., (h,ctx) => 2.0 - generation*(2.0/MaxGen))  // a décroît 2 -> 0
    .WithParam("A",  ..., (h,ctx,a) => 2*a*rnd - a)                  // Eq. (2.3)
    .WithParam("C",  ..., (h,ctx) => 2*rnd)                          // Eq. (2.4)
    .WithParam("l",  ..., (h,ctx,a2) => (a2-1)*rnd + 1.0)            // angle spirale
    .WithCaseGenerator((h,ctx) => rnd.GetDouble() < 0.5)             // tirage principal
    .WithTrue(                                        // POURSUITE (encerclement)
        new IfElseMetaHeuristic()
          .WithCaseGenerator("a", (h,ctx,a) => Math.Abs(a) > 1)
          .WithTrue(new MatchMetaHeuristic()          // |a|>1 : cible ALÉATOIRE (exploration)
              .WithMatches(Current, Random)
              .WithCrossoverMetaHeuristic(encircling))// croisement linéaire D=|C·x-A|
          .WithFalse(new MatchMetaHeuristic()         // |a|<=1 : cible MEILLEURE (exploitation)
              .WithMatches(Current, Best)
              .WithCrossoverMetaHeuristic(encircling)))
    .WithFalse(new MatchMetaHeuristic()               // BUBBLE-NET : spirale hélicoïdale
        .WithMatches(Current, Best)
        .WithCrossoverMetaHeuristic(bubbleNet));       // D'·exp(b·l)·cos(2π·l)+best
```

Chaque ligne est une **primitive** : `IfElseMetaHeuristic` (dispatch booléen), `MatchMetaHeuristic` (choix des parents par `MatchingKind.Current/Random/Best`), `CrossoverMetaHeuristic` + `GeometricCrossover` (l'opérateur de déplacement continu), `WithParam` (paramètres dépendants du contexte : génération, individu). Aucune boîte noire — tout est lisible et modifiable. C'est la réponse de MetaGeneticSharp à Sorensen : on ne nie pas que WOA est une composition d'opérateurs standards, on l'**expose** comme telle.


In [4]:
// --- WOA : raccourci factory + preuve d'équivalence avec la reconstruction ---
IMetaHeuristic woaViaFactory = BuildViaFactory("WhaleOptimisation");
Type woaRootViaFactory = MetaHeuristicsService.GetMetaHeuristicTypeByName("WhaleOptimisation");

// La reconstruction lit le type racine produit par BuildMainHeuristic() : IfElseMetaHeuristic.
Console.WriteLine("WOA via factory            : " + woaViaFactory.GetType().Name);
Console.WriteLine("Type racine (GetMetaHeuristicTypeByName) : " + woaRootViaFactory.Name);
Console.WriteLine("  == typeof(IfElseMetaHeuristic) ? " + (woaRootViaFactory == typeof(IfElseMetaHeuristic)));
Console.WriteLine("  -> la factory renvoie bien l'arbre IfElse de la reconstruction ci-dessus.");
Console.WriteLine();

// Exécution : WOA sur Sphere (unimodale lisse, terrain favorable au bubble-net).
double woaSphere = RunBenchmark(new SphereFitness(), typeof(SphereFitness), woaViaFactory);
Console.WriteLine("WOA sur Sphere (5D, " + Generations + " gen) -> fitness = " + woaSphere.ToString("G5") + "  (proche de 0 = bon)");


WOA via factory            : IfElseMetaHeuristic


Type racine (GetMetaHeuristicTypeByName) : IfElseMetaHeuristic


  == typeof(IfElseMetaHeuristic) ? True


  -> la factory renvoie bien l'arbre IfElse de la reconstruction ci-dessus.


WOA sur Sphere (5D, 80 gen) -> fitness = -2,3658E-10  (proche de 0 = bon)


## 2. Equilibrium Optimizer (EO)

### Description

EO (Faramarzi et al., 2019) est inspiré du **bilan de masse en volume de contrôle** : on estime l'état d'équilibre d'un système dynamique. Le fork le porte directement depuis `mealpy.physics_based.EO`.

La mécanique : on garde un **pool d'équilibre** des 4 meilleures solutions + leur **centroïde**. Chaque particule se déplace vers ce point d'équilibre pondéré, avec un terme de **décroissance exponentielle** du premier ordre qui ralentit le déplacement au fil des générations (taux `f = a1·sign(r-0.5)·(exp(-lambda·t) - 1)`). Un **generation control probability** `gcp` active ou non la mise à jour, ce qui mélange exploitation (vers l'équilibre) et diversité.

### Reconstruction depuis les primitives

`EquilibriumOptimizer.BuildMainHeuristic()` :

```csharp
var eo = new MatchMetaHeuristic()                  // composé = un Match principal
    .WithMatches(Current, Custom)                  // parent + cible "custom" (le centroïde)
    .WithCustomMatchStep(Best, AdditionalPicks=3)  // les 4 meilleures solutions (1+3)
    .WithChildMetaHeuristic(centroidHeuristic)     // GeometricCrossover 4-parents = moyenne (centroïde)
    .WithCrossoverMetaHeuristic(generationHeuristic) // opérateur EO : éq. bilan de masse + decay exp
    .WithParam("t",      (h,ctx) => GetTime(...))   // temps sans dimension Eq. 9
    .WithParam("lambda", (h,ctx,n) => rnd x n genes)// par gène
    .WithParam("f",      (h,ctx,t,l,r) => ExponentialRate) // décroissance Eq. 11
    .WithParam("gcp",    (h,ctx,n,r1,r2) => GetGCP); // generation control Eq. 15
```

L'idée géométrique est limpide : `MatchMetaHeuristic` choisit les parents (les 4 meilleures + le centroïde calculé par un croisement 4-parents), puis le `GeometricCrossover` applique l'équation du bilan de masse. Encore une fois, l'algorithme « physique » se révèle être une composition d'opérateurs standards sur un croisement géométrique.


In [5]:
// --- EO : raccourci factory + exécution ---
IMetaHeuristic eoViaFactory = BuildViaFactory("EquilibriumOptimizer");
Type eoRoot = MetaHeuristicsService.GetMetaHeuristicTypeByName("EquilibriumOptimizer");
Console.WriteLine("EO via factory  : " + eoViaFactory.GetType().Name);
Console.WriteLine("Type racine EO  : " + eoRoot.Name + "  (== MatchMetaHeuristic ? " + (eoRoot == typeof(MatchMetaHeuristic)) + ")");

double eoSphere = RunBenchmark(new SphereFitness(), typeof(SphereFitness), eoViaFactory);
Console.WriteLine("EO sur Sphere -> fitness = " + eoSphere.ToString("G5"));
Console.WriteLine();


EO via factory  : MatchMetaHeuristic


Type racine EO  : MatchMetaHeuristic  (== MatchMetaHeuristic ? True)


EO sur Sphere -> fitness = -2,222E-25


## 3. Forensic-Based Investigation (FBI)

### Description

FBI (Chou & Nguyen, 2020) mime le **processus d'enquête policière** : localisation du suspect, enquête, localisation de la poursuite. Le fork le porte depuis `mealpy.human_based.FBIO`. C'est un composé **multi-phase** : 4 étapes (A1, A2, B1, B2) cyclées à chaque génération.

- **A1** (localisation) : on perturbe un gène aléatoire d'un individu avec un bruit gaussien combiné à deux témoins aléatoires.
- **A2** (enquête) : selon une probabilité liée au rang de fitness de l'individu, on croise avec le meilleur + 3 témoins (5-parent), sinon on ne fait rien (`NoOp`).
- **B1** : combinaison linéaire entre courant et meilleur.
- **B2** : combinaison courant + aléatoire + meilleur, avec une branche dépendant d'un drapeau `randomBetter`.

### Reconstruction depuis les primitives

`ForensicBasedInvestigation.BuildMainHeuristic()` — c'est l'exemple le plus parlant de « composition de phases » :

```csharp
var a1 = new MatchMetaHeuristic().WithMatches(Current, Random, Random)   // 3-parent + bruit gaussien
                                  .WithCrossoverMetaHeuristic(A1Crossover);
var a2 = new IfElseMetaHeuristic()                                        // enquête conditionnelle
            .WithCaseGenerator((h,ctx,prob) => rnd > prob)
            .WithTrue(Match(Current,Best,Random,Random,Random))           // 5-parent
            .WithFalse(new NoOpMetaHeuristic());                          // rien sinon
var b1 = new MatchMetaHeuristic().WithMatches(Current, Best).WithCrossoverMetaHeuristic(B1Crossover);
var b2 = new MatchMetaHeuristic().WithMatches(Current, Random, Best).WithCrossoverMetaHeuristic(B2Crossover);

var fbi = new GenerationMetaHeuristic(1, a1, a2, b1, b2)   // <-- 4 phases cyclées
              .WithScope(EvolutionStage.Crossover);
```

La primitive clef ici est **`GenerationMetaHeuristic`** : elle cycle ses sous-heuristiques par numéro de génération (cf. MGS-2). Un algorithme « humain » à 4 phases devient l'enchaînement de 4 `MatchMetaHeuristic`/`IfElseMetaHeuristic` standards. Changer une phase (par ex. remplacer le bruit gaussien de A1) est une édition locale — impossible dans une `mealpy.FBIO()` monolithique.


In [6]:
// --- FBI : raccourci factory + exécution ---
IMetaHeuristic fbiViaFactory = BuildViaFactory("ForensicBasedInvestigation");
Type fbiRoot = MetaHeuristicsService.GetMetaHeuristicTypeByName("ForensicBasedInvestigation");
Console.WriteLine("FBI via factory : " + fbiViaFactory.GetType().Name);
Console.WriteLine("Type racine FBI : " + fbiRoot.Name + "  (== GenerationMetaHeuristic ? " + (fbiRoot == typeof(GenerationMetaHeuristic)) + ")");

double fbiSphere = RunBenchmark(new SphereFitness(), typeof(SphereFitness), fbiViaFactory);
Console.WriteLine("FBI sur Sphere -> fitness = " + fbiSphere.ToString("G5"));
Console.WriteLine();


FBI via factory : GenerationMetaHeuristic


Type racine FBI : GenerationMetaHeuristic  (== GenerationMetaHeuristic ? True)


FBI sur Sphere -> fitness = -3,3816E-24


## 4. Islands : la composition de composés

### Description

Le modèle insulaire (déjà détaillé en MGS-4) partitionne la population en sous-populations (« îles ») évoluées indépendamment, avec une **migration** périodique qui échange des individus. Le catalogue en expose deux variantes riches :

- **`Islands5Default`** : 5 îles homogènes, toutes en `DefaultMetaHeuristic`, migration « ring » à 2 % (`MediumMigrationRate`). Une variante `NoMigration` désactive les échanges (îles isolées).
- **`Islands5BestMixture`** : 5 îles **hétérogènes** — 2 îles WOA, 2 îles EO, 1 île GA — migration plus fine (0.5 %). C'est le cas d'école de la **composition de composés** : on mélange des algorithmes complets dans un même archipel.

### Reconstruction

Aucune magie : la factory instancie un `IslandCompoundMetaheuristic` (lui-même un `IslandMetaHeuristic`) avec les composés voulus :

```csharp
// Islands5Default : 5 îles identiques (GA), migration ring 2%.
var islandCompound = new IslandCompoundMetaheuristic(
    populationSize / 5, 5, new SimpleCompoundMetaheuristic(new DefaultMetaHeuristic()));
islandCompound.GlobalMigrationRate = IslandMetaHeuristic.MediumMigrationRate;

// Islands5BestMixture : archipel hétérogène 2 WOA + 2 EO + 1 GA.
var mix = new IslandCompoundMetaheuristic(populationSize,
    (2, woaIsland),
    (2, eoIsland),
    (1, woaIsland));
```

On voit ici tout l'intérêt de l'approche primitives-based : un hybride « îles WOA+EO » qui n'existe dans **aucune** bibliothèque monolithique s'énonce en quelques lignes, parce que WOA et EO sont eux-mêmes des composés réutilisables. C'est exactement le prolongement annoncé en conclusion de MGS-6.


In [7]:
// --- Islands : raccourci factory (homogène + hétérogène) + exécution ---
IMetaHeuristic islandsHomog = BuildViaFactory("Islands5Default");
Type islandsRoot = MetaHeuristicsService.GetMetaHeuristicTypeByName("Islands5Default");
Console.WriteLine("Islands5Default via factory : " + islandsHomog.GetType().Name);
Console.WriteLine("Type racine Islands         : " + islandsRoot.Name + "  (== IslandMetaHeuristic ? " + (islandsRoot == typeof(IslandMetaHeuristic)) + ")");

double islSphere = RunBenchmark(new SphereFitness(), typeof(SphereFitness), islandsHomog);
Console.WriteLine("Islands5Default sur Sphere -> fitness = " + islSphere.ToString("G5"));

// Variante hétérogène : 2 WOA + 2 EO + 1 GA (composition de composés).
IMetaHeuristic islandsMix = BuildViaFactory("Islands5BestMixture");
double mixSphere = RunBenchmark(new SphereFitness(), typeof(SphereFitness), islandsMix);
Console.WriteLine("Islands5BestMixture sur Sphere -> fitness = " + mixSphere.ToString("G5"));
Console.WriteLine();


Islands5Default via factory : IslandMetaHeuristic


Type racine Islands         : IslandMetaHeuristic  (== IslandMetaHeuristic ? True)


Islands5Default sur Sphere -> fitness = -0,0048562


Islands5BestMixture sur Sphere -> fitness = -2,1498E-17


## Vue d'ensemble : les quatre familles sur le banc

Pour comparer les composés géométriques entre eux (et avec les îles), nous lançons chacun sur les mêmes fonctions test. Rappel : fitness = objectif négativé, donc la valeur la **moins négative** (la plus proche de 0) est la meilleure. L'objectif n'est pas de désigner un « gagnant » (No Free Lunch : il n'y en a pas), mais de vérifier que **chaque composé reconstruit depuis des primitives est un solveur fonctionnel et compétitif** — la preuve que la reconstruction préserve le pouvoir d'optimisation.


In [8]:
// Grille de comparaison : Default + 4 familles de composés x 3 fonctions test.
var configs = new (string Name, Func<IMetaHeuristic> Build)[]
{
    ("Default (GA)",    () => BuildViaFactory("Default")),
    ("WOA",             () => BuildViaFactory("WhaleOptimisation")),
    ("EO",              () => BuildViaFactory("EquilibriumOptimizer")),
    ("FBI",             () => BuildViaFactory("ForensicBasedInvestigation")),
    ("Islands5BestMix", () => BuildViaFactory("Islands5BestMixture")),
};

string header = "Fonction     " + string.Join("", configs.Select(c => c.Name.PadLeft(16)));
Console.WriteLine(header);
Console.WriteLine(new string('-', 12 + 16 * configs.Length));
foreach (var (fnName, fnType) in demoFns)
{
    var fitness = (IFitness)Activator.CreateInstance(fnType);
    string row = fnName.PadRight(12);
    foreach (var (_, build) in configs)
    {
        double f = RunBenchmark(fitness, fnType, build());
        row += f.ToString("G5").PadLeft(16);
    }
    Console.WriteLine(row);
}
Console.WriteLine();
Console.WriteLine("Chaque colonne est le même type d'objet (IMetaHeuristic) construit en un appel de factory.");


Fonction         Default (GA)             WOA              EO             FBI Islands5BestMix


--------------------------------------------------------------------------------------------


Sphere            -0,0088716     -7,6366E-13     -4,6894E-28     -3,4317E-25     -8,9913E-16


Rastrigin           -0,75004          -7,981              -0              -0     -9,8275E-11


Rosenbrock           -57,953         -4,8681         -3,2756         -3,5385         -3,4638


Chaque colonne est le même type d'objet (IMetaHeuristic) construit en un appel de factory.


## Conclusion : la factory ne cache rien, elle conditionne

Le parcours en trois temps montre la thèse de MetaGeneticSharp en acte :

- **Description** (métaphore) -> **reconstruction** (`BuildMainHeuristic`, code réel lu) -> **factory** (`CreateMetaHeuristicByName`). Les trois désignent **le même objet** : la factory renvoie l'arbre de primitives que la reconstruction assemble (prouvé par `GetMetaHeuristicTypeByName`, qui retourne le type racine `IfElse`/`Match`/`Generation`/`Island`).

La factory n'est donc pas une boîte noire qui *remplacerait* l'algorithme par une magie cachée : c'est un **raccourci vers la reconstruction elle-même**. Si vous voulez comprendre WOA, vous lisez `WhaleOptimisationAlgorithm.BuildMainHeuristic()` — il est là, en plein jour. Si vous voulez l'utiliser, vous appelez la factory. Et si vous voulez le **modifier** (changer l'opérateur bubble-net, ajouter une borne, hybrider avec EO sur des îles), vous éditez une primitive — pas un monolithe.

C'est la différence avec une `mealpy.WOA()` : mealpy offre le raccourci (et c'est précisément le modèle de fiche descriptive que nous avons repris en étape 1), mais n'offre pas la transparence. MetaGeneticSharp offre les deux, parce que le raccourci *est* la reconstruction, exposée.

Le notebook suivant, **MGS-6 Benchmarks**, pousse l'argument jusqu'au bout : mesurer que ces composés reconstruits sont compétitifs face au GA nu et au modèle insulaire, sur 10 fonctions canoniques.


## Exercices

> **Convention** : cellules à compléter. Squelette fourni, à vous d'implémenter. Ne pas lever d'erreur — utiliser `// TODO` et un `Console.WriteLine` informatif.


### Exercice 1 : `WhaleOptimisationNaive` — la variante simplifiée

Le catalogue contient une seconde WOA, `WhaleOptimisationNaive`. La factory l'obtient en remplaçant l'opérateur bubble-net hélicoïdal par une **simple combinaison convexe** (`GetSimpleBubbleNetOperator(mixCoef: 0.5)`, cf. la classe `WhaleOptimisationAlgorithm`). Construisez-la via la factory, lancez-la sur Sphere ET Rastrigin, et comparez à la WOA complète. *Question* : sur quelle famille de surfaces le bubble-net hélicoïdal apporte-t-il un gain par rapport à la combinaison convexe naïve ?


In [9]:
// Exercice 1 : comparer WOA complète vs WhaleOptimisationNaive sur Sphere + Rastrigin.
// TODO étudiant : construire les deux composés via BuildViaFactory, lancer RunBenchmark sur
// Sphere et Rastrigin, et afficher un tableau (complète, naïve) x (Sphere, Rastrigin).

Console.WriteLine("Exercice à compléter : WOA complète vs WhaleOptimisationNaive.");


Exercice à compléter : WOA complète vs WhaleOptimisationNaive.


### Exercice 2 : composer un hybride qui n'existe dans aucune bibliothèque

Toute la valeur de l'approche primitives-based est de rendre les hybrides **triviaux à énoncer**. En vous inspirant de la reconstruction FBI (qui cycle 4 phases avec `GenerationMetaHeuristic`), composez une métaheuristique qui applique **EO pendant la première moitié des générations** puis **WOA pendant la seconde moitié** (un raffinement « exploration équilibre -> exploitation baleine »). *Étape 1* : construire `eoMh` et `woaMh` via la factory. *Étape 2* : les envelopper dans un `new GenerationMetaHeuristic(Generations / 2, eoMh, woaMh)`. *Étape 3* : la lancer via `RunBenchmark` sur Rosenbrock et comparer aux composés seuls.


In [10]:
// Exercice 2 : hybride EO-then-WOA via GenerationMetaHeuristic.
// TODO étudiant :
//   IMetaHeuristic eoMh  = ...;  // BuildViaFactory("EquilibriumOptimizer")
//   IMetaHeuristic woaMh = ...;  // BuildViaFactory("WhaleOptimisation")
//   var hybrid = new GenerationMetaHeuristic(Generations / 2, eoMh, woaMh);
//   double f = RunBenchmark(new RosenbrockFitness(), typeof(RosenbrockFitness), hybrid);

Console.WriteLine("Exercice à compléter : hybride EO-then-WOA (aucune lib ne fournit cela).");


Exercice à compléter : hybride EO-then-WOA (aucune lib ne fournit cela).


### Exercice 3 : preuve d'équivalence factory <-> classe

Pour un composé géométrique au choix (EO ou FBI), vérifiez explicitement que `MetaHeuristicsService.CreateMetaHeuristicByName(name)` et l'appel direct `new XClass().Build()` renvoient **le même type racine** (via `GetMetaHeuristicTypeByName`). C'est la preuve que la factory ne fait qu'envelopper la reconstruction de la classe — pas de magie cachée. *Étape 1* : lire le type racine via `GetMetaHeuristicTypeByName`. *Étape 2* : construire l'instance via la classe (par ex. `new EquilibriumOptimizer { MaxGenerations = Generations }.Build()` après avoir câblé un convertisseur). *Étape 3* : comparer les deux `Type`.


In [11]:
// Exercice 3 : factory == classe (même type racine).
// TODO étudiant : pour EO (ou FBI), comparer :
//   - MetaHeuristicsService.GetMetaHeuristicTypeByName("EquilibriumOptimizer")
//   - le type produit par new EquilibriumOptimizer { MaxGenerations = Generations }.Build()
// (pensez à câbler un GeometricConverter<double> identité, comme la factory le fait en interne).

Console.WriteLine("Exercice à compléter : équivalence factory <-> classe (même type racine).");


Exercice à compléter : équivalence factory <-> classe (même type racine).


## Liens

- [MGS-1 Introduction](MGS-1-Introduction.ipynb) — le moteur autonome `MetaGeneticAlgorithm`
- [MGS-2 Composition](MGS-2-Composition.ipynb) — primitives `Match`, `IfElse` et grammaire fluente
- [MGS-3 Eukaryote](MGS-3-Eukaryote.ipynb) — sous-populations spécialisées (primitive composée)
- [MGS-4 Islands](MGS-4-Islands.ipynb) — modèle insulaire et migration
- [MGS-6 Benchmarks](MGS-6-Benchmarks.ipynb) — comparatif empirique des composés sur 10 fonctions
- [Point d'entrée Part 4](README.md) — positionnement MGS vs GeneticSharp/mealpy
- [Fork jsboige/MetaGeneticSharp](https://github.com/jsboige/MetaGeneticSharp) — code source, `MetaHeuristicsService.cs`, classes composées (`WhaleOptimisationAlgorithm`, `EquilibriumOptimizer`, `ForensicBasedInvestigation`)
